# Interacting with Large Language Models using Semantic Kernel

This notebook demonstrates how to interact with large language models (LLMs) using the Microsoft Semantic Kernel in .NET..

**Objectives:**
- Understand how to set up Semantic Kernel for LLM interaction in .NET.
- Learn to configure and connect to different model providers (OpenAI, Azure OpenAI, GitHub models).
- Send prompts to LLMs and receive responses.
- Use prompt templates and kernel arguments for dynamic, reusable prompts.

## Setup

In this section, we will set up the Semantic Kernel environment and configure it to use different LLM providers.

**Step 1**: Install NuGet packages

To get started with Semantic Kernel, you need to install the required NuGet packages. These packages provide the core functionality for interacting with AI models and managing environment variables. Specifically:
- `Microsoft.SemanticKernel` enables you to build and run AI-powered workflows.
- `DotNetEnv` allows you to load environment variables from a `.env` file, making it easier to manage secrets and configuration settings.

In [1]:
// Import Semantic Kernel
#r "nuget: Microsoft.SemanticKernel, 1.23.0"
#r "nuget: DotNetEnv, 3.1.0"

Installed Packages DotNetEnv, 3.1.0 Microsoft.SemanticKernel, 1.23.0

**Step 2**: Read environment variables

  In this step, we load these variables from a `.env` file (if present) so that they can be accessed by the application.


In [2]:
using DotNetEnv;
using System.IO;

var envFilePath = Path.Combine(Environment.CurrentDirectory, "../..", ".env");
if (File.Exists(envFilePath))
{
    Env.Load(envFilePath);
    Console.WriteLine($"Loaded environment variables from {envFilePath}");
}
else
{
    Console.WriteLine($"No .env file found at {envFilePath}");
}

Loaded environment variables from d:\personal\aiagent-workshop\notebooks\semantic-kernel-basic\../..\.env


**Step 3**: Instantiate the Kernel

The Semantic Kernel is the core component that orchestrates AI services and plugins. In this step, we create and configure a Kernel instance, which will be used to interact with AI models.

In [3]:
using System.ClientModel;
using OpenAI;
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.ChatCompletion;
using System.Text;

OpenAIClient client = null;
if(Environment.GetEnvironmentVariable("USE_AZURE_OPENAI") == "true")
{
    // Configure Azure OpenAI client
    var azureEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT");
    var apiKey = Environment.GetEnvironmentVariable("AZURE_OPENAI_API_KEY");
    client = new OpenAIClient(new ApiKeyCredential(apiKey), new OpenAIClientOptions { Endpoint = new Uri(azureEndpoint) });
}
else if(Environment.GetEnvironmentVariable("USE_OPENAI") == "true")
{
    // Configure OpenAI client
    var apiKey = Environment.GetEnvironmentVariable("OPENAI_API_KEY");
    client = new OpenAIClient(new ApiKeyCredential(apiKey));
}
else if(Environment.GetEnvironmentVariable("USE_GITHUB") == "true")
{
    // Configure GitHub model client
    var uri = Environment.GetEnvironmentVariable("GITHUB_MODEL_ENDPOINT");
    var apiKey = Environment.GetEnvironmentVariable("GITHUB_TOKEN");
    client = new OpenAIClient(new ApiKeyCredential(apiKey), new OpenAIClientOptions { Endpoint = new Uri(uri) });
}

var modelId = Environment.GetEnvironmentVariable("MODEL");
// Create a chat completion service
var builder = Kernel.CreateBuilder();
builder.AddOpenAIChatCompletion(modelId, client);

// Get the chat completion service
Kernel kernel = builder.Build();
var chat = kernel.GetRequiredService<IChatCompletionService>(); 

## Calling LLMs

This section demonstrates how to call different LLMs using the Semantic Kernel.

**Step 4**: Call the Kernel

In this step, we send a prompt to the Semantic Kernel and receive a response from the AI model. 

In [4]:
using Microsoft.SemanticKernel.Connectors.OpenAI;

//call the kernel to get a response
var prompt = "What is the capital of France?";
var response = await kernel.InvokePromptAsync(prompt);
Console.WriteLine($"Response: {response}");

Response: The capital of France is **Paris**.


**Step 5**: Use Prompt Templates and Kernel Arguments

Prompt templates allow you to create reusable prompts with placeholders for dynamic values. Kernel arguments let you pass values to these placeholders at runtime, making your prompts flexible and powerful.

In [ ]:
// Define a prompt template with a placeholder
string template = "What is the capital of {{$country}}?";

// Create a function from the prompt template
var promptTemplateConfig = new PromptTemplateConfig(template);
var promptTemplateFactory = new KernelPromptTemplateFactory();
var promptTemplate = promptTemplateFactory.Create(promptTemplateConfig);
var capitalFunction = kernel.CreateFunctionFromPrompt(template);

// Prepare kernel arguments
var arguments = new KernelArguments
{
    ["country"] = "Japan"
};

// Call the kernel with the function and arguments
var capitalResponse = await kernel.InvokeAsync(capitalFunction, arguments);
Console.WriteLine($"Response: {capitalResponse}");

**Step 6**: Try with another country

You can reuse the same prompt template and function with different arguments to get answers for other countries.

In [ ]:
arguments["country"] = "Brazil";
capitalResponse = await kernel.InvokeAsync(capitalFunction, arguments);
Console.WriteLine($"Response: {capitalResponse}");